# 05 · Dimensiones: claves e historia SCD

**Objetivo:** distinguir SCD1 de SCD2 y explicar la resolución temporal de una clave.

**Modalidad:** material para explicar en clase. Los ejemplos se ejecutan únicamente en memoria, con datos ficticios. No conectan a una base, no crean archivos ni ejecutan una ingesta. Los bloques SQL son ilustrativos y no se envían a ningún servidor.

**Prerrequisitos:** Python con pandas. Cada notebook es independiente: no requiere scripts `.py` ni haber ejecutado otra notebook.


In [ ]:
# Importar conexión centralizada desde el módulo
import sys
sys.path.insert(0, '..')

from conexion_sqlserver import obtener_engines, SERVIDOR, BD_ORIGEN, BD_DESTINO, USUARIO

# Obtener los engines
engine_origen, engine_destino = obtener_engines()
print(f'Conexiones establecidas correctamente.')
print(f'Servidor: {SERVIDOR}')
print(f'Origen: {BD_ORIGEN}')
print(f'Destino: {BD_DESTINO}')


## Modelo Kimball de referencia

`dim_cliente` y `dim_producto`: SCD2 para atributos historizables.
`dim_vendedor` y `dim_ciudad`: SCD1 en este ejemplo.
`dim_tiempo`: calendario.

```text
NK cliente 1
  SK 101: categoría Minorista [2026-01-01, 2026-06-01)
  SK 102: categoría Mayorista [2026-06-01, abierto)
```

Intervalos semiabiertos evitan doble coincidencia en el instante de cambio.
Cambiar un catálogo auxiliar puede modificar la dimensión dependiente.


In [ ]:
import pandas as pd

dimension = pd.DataFrame([
    {"sk": 101, "nk": 1, "categoria": "Minorista",
     "desde": pd.Timestamp("2026-01-01"), "hasta": pd.Timestamp("2026-06-01")},
    {"sk": 102, "nk": 1, "categoria": "Mayorista",
     "desde": pd.Timestamp("2026-06-01"), "hasta": pd.NaT},
])
momento = pd.Timestamp("2026-06-01")
coincidencias = dimension.loc[
    (dimension["nk"] == 1) & (dimension["desde"] <= momento)
    & (dimension["hasta"].isna() | (momento < dimension["hasta"]))
]
if len(coincidencias) != 1:
    raise ValueError("Se requiere una sola versión para resolver la SK")
display(dimension)
print("SK en el cambio:", int(coincidencias.iloc[0]["sk"]))


In [ ]:
# SCD1: corrige la descripción sin conservar otra versión.
ciudades = pd.DataFrame([{"sk": 1, "nk": 10, "nombre": "Ciudad demmo"}])
ciudades.loc[ciudades["nk"] == 10, "nombre"] = "Ciudad demo"
display(ciudades)

# Calendario mínimo de ejemplo, no una expansión ilimitada.
calendario = pd.DataFrame({"fecha": pd.date_range("2016-01-01", "2016-01-07")})
calendario["id_tiempo"] = calendario["fecha"].dt.strftime("%Y%m%d").astype(int)
display(calendario)


## Historia disponible versus historia imaginada

Estos intervalos son ficticios y se proporcionan para explicar la búsqueda.
Un snapshot actual no permite reconstruir la categoría real de un cliente
en 2016. Para eso hacen falta historia del origen o una política explícita
de historia por observación.

No mezclar fecha de factura con vigencia por observación. En producción la
estrategia de resolución temporal debe acordarse antes de cargar hechos.
El calendario debe tener límites de negocio o recursos para fechas extremas.


## Actividad de discusión

¿Qué ocurre si dos versiones tienen vigencias superpuestas? **Respuesta:** la NK puede resolver a dos SK; se bloquea y se revisa el historial.